In [63]:
from lcdb.workflow import PreprocessedWorkflow
from lcdb.workflow.sklearn import *
from lcdb.workflow.xgboost import *
from lcdb.workflow.keras import DenseNNWorkflow
from ConfigSpace.hyperparameters import CategoricalHyperparameter, UniformFloatHyperparameter, UniformIntegerHyperparameter, Constant
import pandas as pd

In [64]:
def get_hpspace(workflow_class, skip_preprocessing_params=True):
    rows = []
    for hp in workflow_class.config_space().values():
        name = hp.name
        if skip_preprocessing_params and name.startswith("pp@"):
            #print(f"Skipping {name}, because this is a pre-processing hyperparameter")
            continue
        rng = None
        if isinstance(hp, (UniformFloatHyperparameter, UniformIntegerHyperparameter)):
            dtype = "float" if isinstance(hp, UniformFloatHyperparameter) else  "int"
            rng = f"[{hp.lower}, {hp.upper}]"
            if hp.log:
                dtype = f"log-{dtype}"
        elif isinstance(hp, CategoricalHyperparameter):
            dtype = "cat"
            rng = str(set(hp.choices)).replace("'", "")
        elif isinstance(hp, Constant):
            pass
        else:
            raise ValueError(f"Unsupported type {type(hp)}")
        rows.append([hp.name, dtype, rng, hp.default_value])
    return pd.DataFrame(rows, columns=["name", "type", "domain", "default_value"])

print(r"\subsubsection{Preprocessors}")
print(get_hpspace(PreprocessedWorkflow, skip_preprocessing_params=False).to_latex(index=False, escape=True))
for workflow_class in [KNNWorkflow, LibLinearWorkflow, LibSVMWorkflow, TreesEnsembleWorkflow, XGBoostWorkflow, DenseNNWorkflow]:
    print(r"\subsubsection{" + str(workflow_class.__name__) + "}")
    print(get_hpspace(workflow_class).to_latex(index=False, escape=True))

\subsubsection{Preprocessors}
\begin{tabular}{llll}
\toprule
name & type & domain & default\_value \\
\midrule
cat\_encoder & cat & \{ordinal, onehot\} & onehot \\
decomposition & cat & \{ka\_rbf, kernel\_pca, none, agglomerator, ka\_nystroem, fastica, lda\} & none \\
featuregen & cat & \{poly, none\} & none \\
featureselector & cat & \{selectp, none\} & none \\
scaler & cat & \{minmax, std, none\} & none \\
kernel\_pca\_kernel & cat & \{rbf, linear\} & linear \\
kernel\_pca\_n\_components & float & [0.25, 1.0] & 1.000000 \\
poly\_degree & float & NaN & 2 \\
selectp\_percentile & int & [25, 100] & 100 \\
std\_with\_std & cat & \{False, True\} & True \\
\bottomrule
\end{tabular}

\subsubsection{KNNWorkflow}
\begin{tabular}{llll}
\toprule
name & type & domain & default\_value \\
\midrule
metric & cat & \{cosine, nan\_euclidean, minkowski\} & minkowski \\
n\_neighbors & log-int & [1, 100] & 5 \\
weights & cat & \{uniform, distance\} & uniform \\
p & int & [1, 10] & 2 \\
\bottomrule
\end{t